In [1]:
import pandas as pd

admissions = pd.read_csv("../data/processed/Elective/elective_admissions.csv")
wait_times = pd.read_csv("../data/processed/Elective/elective_wait_times.csv")
specialty = pd.read_csv("../data/processed/Elective/elective_surgical_specialty.csv")
urgency = pd.read_csv("../data/processed/Elective/elective_clinical_urgency.csv")

In [2]:
admissions.head()

,state,metric,year,value
0,ACT,Admissions,2020–21,15348.0
1,ACT,Admissions,2021–22,14033.0
2,ACT,Admissions,2022–23,12640.0
3,ACT,Admissions,2023–24,14979.0
4,ACT,Admissions,2024–25(b),16565.0


In [3]:
states = pd.concat([
    admissions["state"],
    wait_times["state"],
    urgency["state"]
]).drop_duplicates().sort_values()

dim_state = pd.DataFrame({"state": states}).reset_index(drop=True)

dim_state["state_id"] = dim_state.index + 1

dim_state = dim_state[["state_id", "state"]]

dim_state

,state_id,state
0,1,ACT
1,2,NSW
2,3,NT
3,4,Qld
4,5,SA
5,6,Tas
6,7,Vic
7,8,WA


In [4]:
years = pd.concat([
    admissions["year"],
    wait_times["year"],
    specialty["year"]
]).drop_duplicates().sort_values()

dim_year = pd.DataFrame({"year": years}).reset_index(drop=True)

dim_year["year_id"] = dim_year.index + 1

dim_year = dim_year[["year_id", "year"]]

dim_year

,year_id,year
0,1,2020–21
1,2,2021–22
2,3,2022–23
3,4,2023–24
4,5,2024–25
5,6,2024–25(b)


In [5]:
metrics = pd.concat([
    admissions["metric"],
    wait_times["metric"],
    urgency["metric"]
]).drop_duplicates().sort_values()

dim_metric = pd.DataFrame({"metric": metrics}).reset_index(drop=True)

dim_metric["metric_id"] = dim_metric.index + 1

dim_metric = dim_metric[["metric_id", "metric"]]

dim_metric

,metric_id,metric
0,1,Admissions
1,2,"Admissions per 1,000 population"
2,3,Days waited at the 50th percentile
3,4,Days waited at the 90th percentile
4,5,Percentage waited more than 365 days
5,6,Per cent of admissions


In [6]:
dim_specialty = pd.DataFrame({
    "surgical_specialty": specialty["surgical_specialty"].unique()
})

dim_specialty = dim_specialty.sort_values("surgical_specialty").reset_index(drop=True)

dim_specialty["specialty_id"] = dim_specialty.index + 1

dim_specialty = dim_specialty[["specialty_id", "surgical_specialty"]]

dim_specialty

,specialty_id,surgical_specialty
0,1,Cardio-thoracic surgery
1,2,General surgery
2,3,Gynaecology surgery
3,4,Neurosurgery
4,5,Ophthalmology surgery
5,6,Orthopaedic surgery
6,7,Other
7,8,"Otolaryngology, head and neck surgery"
8,9,Paediatric surgery
9,10,Plastic and reconstructive surgery


In [7]:
dim_clinical_urgency = pd.DataFrame({
    "clinical_urgency": urgency["clinical_urgency"].unique()
})

dim_clinical_urgency = dim_clinical_urgency.sort_values("clinical_urgency").reset_index(drop=True)

dim_clinical_urgency["urgency_id"] = dim_clinical_urgency.index + 1

dim_clinical_urgency = dim_clinical_urgency[["urgency_id", "clinical_urgency"]]

dim_clinical_urgency

,urgency_id,clinical_urgency
0,1,Category 1
1,2,Category 2
2,3,Category 3


In [8]:
fact_elective_admissions = admissions.merge(dim_state, on="state") \
                                     .merge(dim_year, on="year") \
                                     .merge(dim_metric, on="metric")

fact_elective_admissions = fact_elective_admissions[
    ["state_id", "year_id", "metric_id", "value"]
]

fact_elective_admissions.head()

,state_id,year_id,metric_id,value
0,1,1,1,15348.0
1,1,2,1,14033.0
2,1,3,1,12640.0
3,1,4,1,14979.0
4,1,6,1,16565.0


In [9]:
fact_elective_admissions

,state_id,year_id,metric_id,value
0,1,1,1,15348.000000
1,1,2,1,14033.000000
2,1,3,1,12640.000000
3,1,4,1,14979.000000
4,1,6,1,16565.000000
...,...,...,...,...
75,8,1,2,34.063029
76,8,2,2,25.819780
77,8,3,2,28.917290
78,8,4,2,29.745038


In [10]:
fact_wait_times = wait_times.merge(dim_state, on="state") \
                            .merge(dim_year, on="year") \
                            .merge(dim_metric, on="metric")

fact_wait_times = fact_wait_times[
    ["state_id", "year_id", "metric_id", "value"]
]

fact_wait_times.head()

,state_id,year_id,metric_id,value
0,1,1,3,49.0
1,1,2,3,43.0
2,1,3,3,49.0
3,1,4,3,50.0
4,1,5,3,54.0


In [11]:
fact_wait_times

,state_id,year_id,metric_id,value
0,1,1,3,49.0000
1,1,2,3,43.0000
2,1,3,3,49.0000
3,1,4,3,50.0000
4,1,5,3,54.0000
...,...,...,...,...
115,8,1,5,5.1769
116,8,2,5,4.9290
117,8,3,5,8.5416
118,8,4,5,5.8087


In [12]:
fact_specialty = specialty.merge(dim_specialty, on="surgical_specialty") \
                          .merge(dim_year, on="year")

fact_specialty = fact_specialty[
    ["specialty_id", "year_id", "value"]
]

fact_specialty.head()

,specialty_id,year_id,value
0,1,1,10993.0
1,8,1,58776.0
2,2,1,154677.0
3,3,1,90169.0
4,4,1,12757.0


In [13]:
fact_specialty.shape

(60, 3)

In [15]:
fact_clinical_urgency = (
    urgency
    .merge(dim_state, on="state")
    .merge(dim_clinical_urgency, on="clinical_urgency")
    .merge(dim_metric, on="metric")
)

fact_clinical_urgency = fact_clinical_urgency[
    ["state_id", "urgency_id", "metric_id", "value"]
]

In [16]:
output_path = "../data/modelled/Elective/"

In [17]:
dim_state.to_csv(output_path + "dim_state.csv", index=False)
dim_year.to_csv(output_path + "dim_year.csv", index=False)
dim_metric.to_csv(output_path + "dim_metric.csv", index=False)
dim_specialty.to_csv(output_path + "dim_specialty.csv", index=False)
dim_clinical_urgency.to_csv(output_path + "dim_clinical_urgency.csv", index=False)

In [18]:
fact_elective_admissions.to_csv(output_path + "fact_elective_admissions.csv", index=False)
fact_wait_times.to_csv(output_path + "fact_wait_times.csv", index=False)
fact_specialty.to_csv(output_path + "fact_specialty.csv", index=False)
fact_clinical_urgency.to_csv(output_path + "fact_clinical_urgency.csv", index=False)

In [19]:
fact_elective_admissions["metric_id"].unique

<bound method Series.unique of 0     1
1     1
2     1
3     1
4     1
     ..
75    2
76    2
77    2
78    2
79    2
Name: metric_id, Length: 80, dtype: int64>

In [20]:
dim_metric

,metric_id,metric
0,1,Admissions
1,2,"Admissions per 1,000 population"
2,3,Days waited at the 50th percentile
3,4,Days waited at the 90th percentile
4,5,Percentage waited more than 365 days
5,6,Per cent of admissions


In [21]:
fact_wait_times

,state_id,year_id,metric_id,value
0,1,1,3,49.0000
1,1,2,3,43.0000
2,1,3,3,49.0000
3,1,4,3,50.0000
4,1,5,3,54.0000
...,...,...,...,...
115,8,1,5,5.1769
116,8,2,5,4.9290
117,8,3,5,8.5416
118,8,4,5,5.8087


# Data Dictionary: Elective Surgery Data Model

## Dimension Tables

### `dim_state`
| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| `state_id` | Integer (PK) | Unique identifier for the Australian state or territory. |
| `state` | String | Name or abbreviation of the state (e.g., NSW, Vic, Qld). |

### `dim_year`
| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| `year_id` | Integer (PK) | Unique identifier for the reporting financial year. |
| `year` | String | Financial year period (e.g., 2020–21, 2021–22). |

### `dim_metric`
| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| `metric_id` | Integer (PK) | Unique identifier for the performance metric. |
| `metric` | String | Description of the metric (e.g., Admissions, Days waited at 50th percentile). |

### `dim_specialty`
| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| `specialty_id` | Integer (PK) | Unique identifier for the surgical specialty. |
| `surgical_specialty` | String | Name of the specialty (e.g., Cardio-thoracic surgery, Neurosurgery). |

### `dim_clinical_urgency`
| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| `urgency_id` | Integer (PK) | Unique identifier for the clinical urgency category. |
| `clinical_urgency` | String | Urgency classification (e.g., Category 1, Category 2, Category 3). |

---

## Fact Tables

### `fact_elective_admissions`
| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| `state_id` | Integer (FK) | Reference to `dim_state`. |
| `year_id` | Integer (FK) | Reference to `dim_year`. |
| `metric_id` | Integer (FK) | Reference to `dim_metric`. |
| `value` | Float | The numerical value of the admission metric. |

### `fact_wait_times`
| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| `state_id` | Integer (FK) | Reference to `dim_state`. |
| `year_id` | Integer (FK) | Reference to `dim_year`. |
| `metric_id` | Integer (FK) | Reference to `dim_metric`. |
| `value` | Float | The waiting time value (days or percentage). |

### `fact_specialty`
| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| `specialty_id` | Integer (FK) | Reference to `dim_specialty`. |
| `year_id` | Integer (FK) | Reference to `dim_year`. |
| `value` | Float | Numerical value associated with the surgical specialty. |

### `fact_clinical_urgency`
| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| `urgency_id` | Integer (FK) | Reference to `dim_clinical_urgency`. |
| `state_id` | Integer (FK) | Reference to `dim_state`. |
| `metric_id` | Integer (FK) | Reference to `dim_metric`. |
| `value` | Float | Numerical value associated with the urgency category. |